# Figure: magnetization transfer (toy model)

The cell below is tagged `#| label: fig-mt-output`, so `04-issues.md` embeds its output with `:::{figure} #fig-mt-output`.

In [ ]:
#| label: fig-mt-output
# Toy model of magnetization transfer (MT): a visible (free water) and an invisible (bound)
# pool of hydrogen that exchange magnetization. The readout pulses partly saturate the
# invisible pool, and through the exchange this also lowers the visible magnetization.
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import warnings, logging
# Kaleido (the PNG export for the PDF) can print harmless clean-up warnings on Windows.
warnings.filterwarnings("ignore", message=".*temporary directory.*")
for name in ("choreographer", "kaleido", "logistro"):
    logging.getLogger(name).setLevel(logging.CRITICAL)
import os, sys
sys.path.append(os.path.join(os.getcwd(), "notebooks"))  # find cardiac_sim.py also when run from the book folder
from cardiac_sim import SEQUENCES, schedule, fit, ll_model, sasha_model

# Interactive figure for the website + PNG snapshot for the PDF (same probe as the template).
try:
    pio.to_image(go.Figure(), format="png")
    pio.renderers.default = "plotly_mimetype+png"
except Exception as exc:
    pio.renderers.default = "plotly_mimetype"
    print(f"Static PNG export unavailable ({type(exc).__name__}); PDF will show the caption only.")

# --- Parameters (toy model, chosen to show the trend, not exact values) -----------------
T1 = 1000.0            # T1 of the visible water, native myocardium [ms]
RR = 1000.0            # 60 beats per minute
T1_BOUND = 1000.0      # T1 of the invisible pool [ms]
K_EXCHANGE = 0.003     # exchange rate from visible to invisible pool [1/ms]
SAT_PER_IMAGE = 0.3    # fraction of the invisible pool saturated by one image (readout)
ALPHA_EFF = 35.0       # each image acts like one 35° tip on the visible pool, as before
MT_SIZES = [0.0, 0.05, 0.10, 0.15, 0.20]   # size of the invisible pool, relative to the visible one
DT = 2.0               # time step [ms]
COLORS = {"MOLLI": "royalblue", "ShMOLLI": "seagreen", "SASHA": "darkorange"}

def simulate_mt(seq, F):
    """Visible (mf) and invisible (mb) longitudinal magnetization with exchange."""
    kf = K_EXCHANGE if F > 0 else 0.0
    kb = kf / F if F > 0 else 0.0                # balance: kf * 1 = kb * F at equilibrium
    ca = np.cos(np.deg2rad(ALPHA_EFF))
    events = schedule(seq, RR)
    mf, mb, t, last, samples = 1.0, F, events[0][0] - 300, None, []
    for t_ev, kind in events:
        for _ in range(int((t_ev - t) / DT)):    # relaxation + exchange until the event
            dmf = (1 - mf) / T1 - kf * mf + kb * mb
            dmb = (F - mb) / T1_BOUND + kf * mf - kb * mb if F > 0 else 0.0
            mf, mb = mf + dmf * DT, mb + dmb * DT
        t = t_ev
        if kind == "inv":
            mf, mb, last = -mf, 0.0, t           # invisible pool cannot be inverted, only saturated
        elif kind == "sat":
            mf, mb, last = 0.0, 0.0, t
        else:
            samples.append((t - last if last is not None else np.inf, mf))
            mf, mb = mf * ca, mb * (1 - SAT_PER_IMAGE)
    return samples

results = {seq: [] for seq in SEQUENCES}
curves = {}
for F in MT_SIZES:
    for seq in SEQUENCES:
        samples = simulate_mt(seq, F)
        model, p, T1_fit, n_used = fit(seq, samples, RR)
        results[seq].append(T1_fit)
        if seq == "MOLLI":
            curves[F] = (samples, p)

# --- Figure ----------------------------------------------------------------------------
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.14, column_widths=[0.55, 0.45])
xf = np.linspace(0, 4500, 300)

# static: MOLLI without MT for comparison (left), fitted T1 vs MT size for all methods (right)
s0, p0 = curves[0.0]
fig.add_trace(go.Scatter(x=xf, y=ll_model(xf, *p0), mode="lines",
                         line=dict(color="lightgray", width=5), name="MOLLI without MT",
                         hoverinfo="skip"), row=1, col=1)
for seq in SEQUENCES:
    fig.add_trace(go.Scatter(x=[100 * F for F in MT_SIZES], y=results[seq], mode="lines",
                             line=dict(color=COLORS[seq], width=3), name=seq,
                             hovertemplate=seq + ": %{y:.0f} ms<extra></extra>"), row=1, col=2)
fig.add_hline(y=T1, line=dict(color="black", dash="dash", width=1), row=1, col=2)
n_static = 1 + len(SEQUENCES)

TRACES = 3
for F in MT_SIZES:
    vis = (F == 0.10)
    samples, p = curves[F]
    x = [s[0] for s in samples]; y = [s[1] for s in samples]
    fig.add_trace(go.Scatter(x=x, y=y, mode="markers", marker=dict(size=7, color="royalblue"),
                             name="MOLLI images with MT", visible=vis,
                             hovertemplate="TI = %{x:.0f} ms<br>Mz = %{y:.2f}<extra></extra>"),
                  row=1, col=1)
    T1_fit = p[2] * (p[1] / p[0] - 1)
    fig.add_trace(go.Scatter(x=xf, y=ll_model(xf, *p), mode="lines",
                             line=dict(color="royalblue", dash="dash"),
                             name=f"MOLLI fit with MT: T1 = {T1_fit:.0f} ms", visible=vis,
                             hoverinfo="skip"), row=1, col=1)
    idx = MT_SIZES.index(F)
    fig.add_trace(go.Scatter(x=[100 * F] * 3, y=[results[s][idx] for s in SEQUENCES],
                             mode="markers", marker=dict(size=11, color=[COLORS[s] for s in SEQUENCES],
                                                         symbol="diamond"),
                             showlegend=False, visible=vis, hoverinfo="skip"), row=1, col=2)

steps = []
for i, F in enumerate(MT_SIZES):
    visible = [True] * n_static + [False] * (TRACES * len(MT_SIZES))
    visible[n_static + i * TRACES: n_static + (i + 1) * TRACES] = [True] * TRACES
    steps.append(dict(method="update", args=[{"visible": visible}], label=f"{100 * F:.0f}%"))

fig.update_layout(
    sliders=[dict(active=MT_SIZES.index(0.10), steps=steps, pad=dict(t=60),
                  currentvalue=dict(prefix="Invisible (bound) hydrogen: "))],
    height=620, template="plotly_white", margin=dict(t=30, b=230),
    # legend at the very bottom of the figure, below the slider, with space in between
    legend=dict(orientation="h", yref="container", yanchor="bottom", y=0.01, x=0),
)
fig.update_xaxes(title_text="TI [ms]", range=[0, 4600], row=1, col=1)
fig.update_yaxes(title_text="Mz of the visible water", range=[-1.1, 1.1], row=1, col=1)
fig.update_xaxes(title_text="Invisible hydrogen [% of visible]", row=1, col=2)
fig.update_yaxes(title_text="Fitted T1 [ms]", range=[750, 1050], row=1, col=2)

# Only meaningful values on hover; no zooming, legend not clickable.
fig.update_layout(hovermode="closest", dragmode=False,
                  legend=dict(itemclick=False, itemdoubleclick=False))
fig.update_xaxes(fixedrange=True)
fig.update_yaxes(fixedrange=True)
fig
